# 그래프를 노드로 꽂기 · subgraph

강의 오후 ④ 뒷부분 (Academy 모듈 4 · L2 Sub-graphs, 개념만)

| 칸 | 보여 주는 것 |
| --- | --- |
| ① 그래프를 노드로 | compile 한 그래프를 부모의 `addNode` 에 그대로 넣는다 |
| ② 무엇이 오가나 | 부모와 자식은 **이름이 같은 필드**만 주고받는다. 자식만의 필드는 자식 안에 남는다 |
| ③ 안쪽 단계 보기 | `stream(…, { subgraphs: true })` 면 자식 안의 단계도 보인다 |

모델은 부르지 않습니다. 사전 찾기만 하는 작은 그래프로 모양만 봅니다.

In [ ]:
import { Annotation, StateGraph, START, END } from "npm:@langchain/langgraph@1.4.18";

const DICT: Record<string, string> = { ubiquitous: "어디에나 있는", meticulous: "꼼꼼한" };

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

## ① 그래프를 노드로 꽂는다

자식 그래프 `dictionary` 는 단어(`word`)를 받아 뜻(`meaning`)을 찾습니다. 몇 번 찾았는지(`tries`)는 자기만 씁니다.

부모 그래프는 질문에서 단어를 고르고(`pick`), 사전을 찾고, 답을 씁니다(`write`). 사전을 찾는 노드 자리에 **compile 한 자식 그래프를 그대로** 넣습니다.

In [ ]:
// 자식 그래프: word → meaning. tries 는 자식 안에서만 쓴다
const DictState = Annotation.Root({
  word: Annotation<string>(),
  meaning: Annotation<string>(),
  tries: Annotation<number>(),
});
const dictionary = new StateGraph(DictState)
  .addNode("look", (s) => {
    console.log("  [자식 look] 받은 필드:", Object.keys(s).filter((k) => (s as any)[k] !== undefined));
    return { tries: 1, meaning: DICT[s.word] ?? "사전에 없음" };
  })
  .addNode("check", (s) => ({ tries: s.tries + 1 }))
  .addEdge(START, "look")
  .addEdge("look", "check")
  .addEdge("check", END)
  .compile();
await showGraph(dictionary);

// 부모 그래프: question → word → (자식) → answer
const TutorState = Annotation.Root({
  question: Annotation<string>(),
  word: Annotation<string>(),
  meaning: Annotation<string>(),
  answer: Annotation<string>(),
});
const tutor = new StateGraph(TutorState)
  .addNode("pick", (s) => ({ word: s.question.split(" ")[0] }))
  .addNode("dictionary", dictionary)   // compile 한 그래프를 노드로 그대로
  .addNode("write", (s) => ({ answer: `${s.word} 는 "${s.meaning}" 이라는 뜻입니다` }))
  .addEdge(START, "pick")
  .addEdge("pick", "dictionary")
  .addEdge("dictionary", "write")
  .addEdge("write", END)
  .compile();
await showGraph(tutor);

console.log(await tutor.invoke({ question: "ubiquitous 뜻이 뭐야?" }));

**돌려 보면** 부모의 결과에 `answer` 까지 나옵니다. 부모 입장에서 `dictionary` 는 그냥 노드 하나입니다.

## ② 무엇이 오가나 — 이름이 같은 필드만

위 출력에서 두 가지를 볼 수 있습니다.

- 자식 `look` 이 받은 필드는 `word` 하나입니다. 부모의 `question` 은 자식 State 에 없는 이름이라 넘어가지 않았습니다.
- 부모의 결과에 `tries` 가 없습니다. 자식만의 필드라 자식 안에 남았습니다. 부모로 올라온 건 이름이 같은 `word` · `meaning` 입니다 (③ 의 `{"dictionary":{"word":…,"meaning":…}}`).

## ③ 안쪽 단계 보기 — subgraphs: true

updates 로 stream 해 보면, 부모 쪽에서는 `dictionary` 가 한 번에 끝난 것으로 보입니다. `subgraphs: true` 를 주면 자식 안의 `look` · `check` 까지 보입니다.

In [ ]:
// ③ 부모의 단계만
for await (const update of await tutor.stream({ question: "meticulous 뜻" }, { streamMode: "updates" })) {
  console.log(JSON.stringify(update));
}

In [ ]:
await showGraph(tutor, { xray: true });   // 그림도 xray 로 그리면 dictionary 안의 look · check 까지 나온다

// subgraphs: true 면 [어느 그래프인지, 값] 으로 온다. [] 는 부모, ["dictionary:…"] 는 자식
for await (const [where, update] of await tutor.stream({ question: "meticulous 뜻" }, { streamMode: "updates", subgraphs: true })) {
  console.log(where.length ? "  자식" : "부모", JSON.stringify(update));
}

**돌려 보면** 그림에서는 `dictionary` 가 상자로 열려 안의 `look` · `check` 가 보입니다. stream 에서도 자식의 `look` · `check` 가 부모의 `dictionary` 보다 먼저 찍힙니다. `tries` 도 보입니다. 받는 값이 `[어느 그래프인지, 값]` 모양이라 부모와 자식을 나눠 찍을 수 있습니다.